# Multiagent Pattern - Program Promotion Warm-Up

This warm-up notebook introduces the Multiagent Pattern in a simple non-forensics setting before you move into the Lab 5 case. Instead of asking one agent to do everything, we give different agents different roles and let their outputs build on one another.

If you are new to this course sequence, it may help to review the earlier labs first:
- [Reflection Pattern](../lab1_reflection_pattern/03_lab_notebook.ipynb)
- [Tool Use Pattern](../lab2_tool_use_pattern/03b_lab_notebook.ipynb)
- [ReAct Pattern](../lab3_react_pattern/03b_lab_notebook.ipynb)
- [Planning Pattern](../lab4_planning_pattern/03a_lab_notebook.ipynb)


<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2f686c08ca-989b-4083-9128-e6bc2a8c07b5_716x526-3.gif" alt="General Multiagent Pattern" width="500"/>

This figure shows the general Multiagent Pattern: different agents handle different parts of the work, and their outputs are combined into one stronger result.

## Learning Goal

By the end of this warm-up, you should be able to explain:
1. why one agent can focus on one role instead of doing everything,
2. how context is passed from one agent to another,
3. and how `Crew` packages the same collaboration workflow in a more organized way.

## What You Will Do
1. Set up the notebook and load the Lab 5 model settings.
2. Read a simple promotion question about the BS Cyber Forensics program at the University of Baltimore.
3. Define three role-specialized agents one at a time.
4. Connect the agents so downstream agents receive upstream context automatically.
5. Run the three agents step by step and inspect how context is passed.
6. Run the same workflow again with `Crew`, which keeps the dependency order organized for you.


## Quick Vocabulary

A few plain-language words before you begin:
- `Agent`: one role-specific helper that works on one part of the problem.
- `Context`: the earlier output that is passed to another agent as working notes.
- `Dependency`: the rule that one agent should wait for another agent's output first.
- `Crew`: the object that organizes a team of agents and runs them in dependency order.

This notebook shows two equally important views of the same workflow: first the collaboration is shown manually, agent by agent; then the same collaboration is shown again using `Crew`.


## Warm-Up Question

Use the following non-forensics question in this notebook:

**How should the University of Baltimore promote its BS Cyber Forensics program over the next three years so it reaches the right students, explains the program's value clearly, and uses realistic outreach methods?**

The three agents in this warm-up have different jobs:
- `AudienceAgent`: decide who the program should target first.
- `ProgramValueAgent`: decide which program strengths matter most.
- `OutreachAgent`: combine those earlier ideas into one three-year promotion plan.

The main learning goal is not marketing. The goal is to see how multiagent collaboration works when each agent has a clear role.


## Expected Agent Outputs

Keep this checklist in mind as you move through the notebook:
- `AudienceAgent` should produce: priority student groups, why each group fits, and audience advice for the team.
- `ProgramValueAgent` should produce: most important program strengths, why students should care, and message themes.
- `OutreachAgent` should produce: a final working three-year promotion plan with `Year 1`, `Year 2`, and `Year 3` outreach focus.

So in this warm-up, the first two agents prepare specialized inputs, and `OutreachAgent` writes the combined final plan.


## Part 1: Build and Run the Collaboration Manually

In Part 1, you will set up the notebook, define the three agents, connect their dependencies, and then run them one at a time. This keeps the collaboration visible, so you can see exactly what each agent contributes before the packaged `Crew` version appears.


### Step 1: Set Up the Notebook

Why this step matters: the rest of the notebook depends on the Lab 5 model settings and the local course code. This setup cell makes sure Python can find the course classes before you create any agents.


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

LAB_NAME = 'lab5_multiagent_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(f'Open this notebook from the {LAB_NAME} folder.')

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError('Expected .env in this folder. Copy .env.example to .env first.')

# Add the local src/ folder so Python can import the course code used in this notebook.
src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

# Import the course-specific classes only after src/ has been added to the Python path.
from agentic_patterns.multiagent_pattern.agent import Agent
from agentic_patterns.multiagent_pattern.crew import Crew

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)
print('Model:', MODEL)


### Step 2: Create the Shared Promotion Brief

Why this step matters: all three agents need the same starting facts. This cell creates one shared promotion question and one short program brief so the team is working from the same information.


In [ ]:
PROMOTION_QUESTION = (
    'Create a three-year plan to promote the University of Baltimore BS Cyber Forensics program to prospective students. '
    'Identify the best student audiences, explain the program strengths that matter most, '
    'and recommend realistic outreach methods.'
)

# This shared brief keeps the example grounded and gives every agent the same basic facts.
PROGRAM_BRIEF = """
Program facts:
- The University of Baltimore BS Cyber Forensics program emphasizes hands-on labs, digital evidence analysis, mobile device analysis, and incident response.
- Graduates may pursue careers in digital forensics, cybersecurity operations, and investigative support roles.
- The University of Baltimore wants to attract high school seniors, community-college transfer students, and adult learners changing careers.
- The promotion budget is moderate, so the plan should favor realistic actions rather than expensive campaigns.
- The plan should cover three years, not just one semester.
""".strip()

print(PROMOTION_QUESTION)
print()
print(PROGRAM_BRIEF)


### Step 3: Define `AudienceAgent`

Why this step matters: the first agent narrows the audience question. It does not try to build the whole plan. Its job is only to decide who should be prioritized.


In [ ]:
# Create the first specialist. It focuses only on target audiences.
audience_agent = Agent(
    name='AudienceAgent',
    backstory=(
        'You identify the best student audiences for academic program promotion. '
        'You focus on who the program should reach first and why.'
    ),
    # task_description = the question this one agent should answer.
    task_description=(
        f'{PROMOTION_QUESTION}\n\n'
        f'{PROGRAM_BRIEF}\n\n'
        'Focus only on the audience question. Identify the 2 or 3 target groups that matter most '
        'for this program and explain why each group is a strong fit.'
    ),
    # task_expected_output = the format we want the agent to follow.
    task_expected_output=(
        'Use these labels:\n'
        'Priority student groups:\n'
        'Why each group fits:\n'
        'Audience advice for the rest of the team:'
    ),
    llm=MODEL,
)

print(audience_agent)


### Step 4: Define `ProgramValueAgent`

Why this step matters: the second agent adds a different kind of thinking. Instead of asking who to reach, it asks what program strengths should be emphasized.


In [ ]:
# Create the second specialist. It focuses on the value of the program.
program_value_agent = Agent(
    name='ProgramValueAgent',
    backstory=(
        'You explain the strongest value of an academic program in student-friendly language. '
        'You focus on program strengths, benefits, and message themes.'
    ),
    task_description=(
        f'{PROMOTION_QUESTION}\n\n'
        f'{PROGRAM_BRIEF}\n\n'
        'Focus only on program value. Explain which strengths of the University of Baltimore BS Cyber Forensics program '
        'should be highlighted most clearly to prospective students.'
    ),
    task_expected_output=(
        'Use these labels:\n'
        'Most important program strengths:\n'
        'Why students should care:\n'
        'Message themes for outreach materials:'
    ),
    llm=MODEL,
)

print(program_value_agent)


### Step 5: Define `OutreachAgent`

Why this step matters: the third agent is the synthesizing agent. It will use the earlier audience and value ideas to build the final three-year working plan.


In [ ]:
# Create the final specialist. It turns earlier ideas into one phased outreach plan.
outreach_agent = Agent(
    name='OutreachAgent',
    backstory=(
        'You create realistic outreach plans for academic programs. '
        'You combine earlier audience and value insights into one practical phased plan.'
    ),
    task_description=(
        f'{PROMOTION_QUESTION}\n\n'
        f'{PROGRAM_BRIEF}\n\n'
        'Use the earlier agent context to create one practical three-year outreach plan. '
        'Include a simple Year 1, Year 2, and Year 3 timeline.'
    ),
    task_expected_output=(
        'Use these labels:\n'
        'Priority audience:\n'
        'Main messages:\n'
        'Year 1 outreach focus:\n'
        'Year 2 outreach focus:\n'
        'Year 3 outreach focus:\n'
        'Final working promotion plan:'
    ),
    llm=MODEL,
)

print(outreach_agent)


### Step 6: Connect the Agents

Why this step matters: this step finishes the manual collaboration setup. The `>>` arrows tell the notebook which agent should pass context to the next one.


In [ ]:
# `>>` means: pass this agent's output to the next agent as context.
audience_agent >> program_value_agent
audience_agent >> outreach_agent
program_value_agent >> outreach_agent

# Show the dependency structure after the arrows are added.
print('AudienceAgent dependents:', audience_agent.dependents)
print('ProgramValueAgent dependencies:', program_value_agent.dependencies)
print('OutreachAgent dependencies:', outreach_agent.dependencies)

print('Part 1 setup is complete.')


### Step 7: Run `AudienceAgent`

Why this step matters: this first run should give you the team's starting audience strategy. You should expect priority student groups and a short explanation of why they fit.


In [ ]:
audience_output = audience_agent.run()
display(Markdown('### AudienceAgent Output\n\n' + audience_output))


`AudienceAgent` has now contributed the first useful layer of the plan: who the program should reach first. The downstream agents can use this as working guidance instead of starting from scratch.


### Step 8: Inspect the Context Passed to `ProgramValueAgent`

Why this step matters: this cell shows the “working memory” passed from one agent to the next. In this notebook, context is simply earlier output that is stored and reused.


In [ ]:
display(Markdown('### ProgramValueAgent Received Context\n\n```text\n' + program_value_agent.context + '\n```'))


### Step 9: Run `ProgramValueAgent`

Why this step matters: this second run should add a new type of information. Instead of repeating the audience list, it should explain which program strengths and message themes matter most for those audiences.


In [ ]:
program_value_output = program_value_agent.run()
display(Markdown('### ProgramValueAgent Output\n\n' + program_value_output))


`ProgramValueAgent` adds the second layer of the plan: what to say and why students should care. At this point, the team has both audience guidance and message guidance.


### Step 10: Inspect the Context Passed to `OutreachAgent`

Why this step matters: now you can see that `OutreachAgent` receives both earlier outputs together. That combined context is what lets it synthesize a final plan instead of guessing alone.


In [ ]:
display(Markdown('### OutreachAgent Received Context\n\n```text\n' + outreach_agent.context + '\n```'))


### Step 11: Run `OutreachAgent`

Why this step matters: this final manual run should transform the earlier audience and value ideas into one staged three-year promotion plan with a simple timeline.


In [ ]:
outreach_output = outreach_agent.run()
display(Markdown('### OutreachAgent Final Promotion Plan\n\n' + outreach_output))


`OutreachAgent` has now turned both earlier outputs into the final working plan. This is the main Multiagent Pattern idea: different roles contribute different pieces, and the final agent combines them into one stronger answer.


## Part 2: Run the Same Workflow with `Crew`

`Crew` does not replace the agents. It organizes the same team and runs them in dependency order. This means Part 2 is not a different workflow. It is the same collaboration pattern packaged in a cleaner way.


### Step 1: Prepare a Fresh Team for `Crew`

Why this step matters: Part 2 should start with a clean copy of the same three-agent team, not the agents that already ran in Part 1. This helper rebuilds that fresh team and also gives `Crew` a small Markdown display helper.


In [ ]:
def build_program_promotion_agents():
    """Create a fresh copy of the same three-agent team for Part 2."""
    audience_agent = Agent(
        name='AudienceAgent',
        backstory=(
            'You identify the best student audiences for academic program promotion. '
            'You focus on who the program should reach first and why.'
        ),
        task_description=(
            f'{PROMOTION_QUESTION}\n\n'
            f'{PROGRAM_BRIEF}\n\n'
            'Focus only on the audience question. Identify the 2 or 3 target groups that matter most '
            'for this program and explain why each group is a strong fit.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Priority student groups:\n'
            'Why each group fits:\n'
            'Audience advice for the rest of the team:'
        ),
        llm=MODEL,
    )

    program_value_agent = Agent(
        name='ProgramValueAgent',
        backstory=(
            'You explain the strongest value of an academic program in student-friendly language. '
            'You focus on program strengths, benefits, and message themes.'
        ),
        task_description=(
            f'{PROMOTION_QUESTION}\n\n'
            f'{PROGRAM_BRIEF}\n\n'
            'Focus only on program value. Explain which strengths of the University of Baltimore BS Cyber Forensics program '
            'should be highlighted most clearly to prospective students.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Most important program strengths:\n'
            'Why students should care:\n'
            'Message themes for outreach materials:'
        ),
        llm=MODEL,
    )

    outreach_agent = Agent(
        name='OutreachAgent',
        backstory=(
            'You create realistic outreach plans for academic programs. '
            'You combine earlier audience and value insights into one practical phased plan.'
        ),
        task_description=(
            f'{PROMOTION_QUESTION}\n\n'
            f'{PROGRAM_BRIEF}\n\n'
            'Use the earlier agent context to create one practical three-year outreach plan. '
            'Include a simple Year 1, Year 2, and Year 3 timeline.'
        ),
        task_expected_output=(
            'Use these labels:\n'
            'Priority audience:\n'
            'Main messages:\n'
            'Year 1 outreach focus:\n'
            'Year 2 outreach focus:\n'
            'Year 3 outreach focus:\n'
            'Final working promotion plan:'
        ),
        llm=MODEL,
    )

    # Recreate the same dependency graph for the Crew part.
    audience_agent >> program_value_agent
    audience_agent >> outreach_agent
    program_value_agent >> outreach_agent

    return audience_agent, program_value_agent, outreach_agent


def run_crew_with_markdown(crew: Crew):
    """Run the crew in dependency order and show each output as readable Markdown."""
    outputs = {}
    for agent in crew.topological_sort():
        output = agent.run()
        outputs[agent.name] = output
        display(Markdown(f'### {agent.name} Output\n\n{output}'))
    return outputs


### Step 2: Build the Same Team Inside `Crew`

Why this step matters: this cell creates a fresh copy of the same three-agent team inside a `Crew` context so the team graph can be stored and reused.


In [ ]:
with Crew() as crew:
    crew_audience_agent, crew_program_value_agent, crew_outreach_agent = build_program_promotion_agents()


### Step 3: Visualize the Crew Graph

Why this step matters: the graph is a picture of the dependency rules you already created manually.

When you inspect the graph, look for this pattern:
- `AudienceAgent` feeds both later agents.
- `ProgramValueAgent` also feeds `OutreachAgent`.
- `OutreachAgent` runs last because it depends on both earlier outputs.


In [ ]:
crew.plot()


### Step 4: Run the Full Crew

Why this step matters: this is the same collaboration pattern as Part 1, but now `Crew` keeps the dependency order organized for you. Compare the outputs with the earlier manual run rather than thinking of this as a different system.


In [ ]:
crew_outputs = run_crew_with_markdown(crew)


## Why This Warm-Up Matters

This notebook is a non-forensics example, but the collaboration idea is the same one you will use in the main Lab 5 case:
- one agent can focus on the first interpretation,
- another can verify whether that interpretation is well supported,
- another can combine the earlier work into a final decision.

Next, move to [03b_lab_notebook.ipynb](./03b_lab_notebook.ipynb) for the full forensic multiagent case.
